[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# The Estimator API &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


**1.** Fit a `LinearRegression` that predicts `jan_mean` from `latitude` and `elevation_m` alone, and print its two coefficients with their feature names.

In [2]:
two = ["latitude", "elevation_m"]
small = LinearRegression().fit(network[two], network["jan_mean"])

print(pd.Series(small.coef_, index=two).round(4))

latitude      -0.5487
elevation_m   -0.0083
dtype: float64


Both coefficients are negative, since further north and higher up are colder. They differ from the
five-feature model's, because each now also stands in for the features left out.

**2.** Predict the January mean of a planned station at latitude 69.5, longitude 23.0, elevation 450 m, 21 km from the coast, installed in 2026, with the `linear` model. Build the row as a one-row DataFrame with the columns in `FEATURES`.

In [3]:
linear = LinearRegression().fit(network[FEATURES], network["jan_mean"])
planned = pd.DataFrame([[69.5, 23.0, 450, 21.0, 2026]], columns=FEATURES)

print(linear.predict(planned).round(1))

[-8.5]


The first line fits `linear` as the worked example did, since this notebook runs on its own. A list holding one list makes a single row, and naming its columns lets the model check them.

**3.** Make a `DecisionTreeRegressor` with `max_depth=2` and `random_state=0`, fit it, and print its `get_depth()` and its `score` on `X` and `y`.

In [4]:
X = network[FEATURES]
y = network["jan_mean"]
shallow = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X, y)

print(shallow.get_depth(), round(shallow.score(X, y), 3))

2 0.579


Two levels of questions split the stations into at most four groups, each predicted by its group's mean, so the tree scores well below the deeper ones.

**4.** Fit a `DecisionTreeClassifier(max_depth=3, random_state=0)` on `june_frost` instead of snow, and print `classes_` and how many of the 1,200 stations it predicts will have a frost in June.

In [5]:
frost = DecisionTreeClassifier(max_depth=3, random_state=0).fit(network[FEATURES], network["june_frost"])

print(frost.classes_)
print(frost.predict(network[FEATURES]).sum(), "of", len(network))

[False  True]
32 of 1200


`predict` returns `True` or `False` for each station, and `sum` counts the `True` values.

**5.** Use `StandardScaler` on `X`, and print the mean and standard deviation of every transformed column, rounded to two places.

In [6]:
scaled = StandardScaler().fit_transform(network[FEATURES])

print("every mean is 0:", np.allclose(scaled.mean(axis=0), 0))
print(scaled.std(axis=0).round(2))

every mean is 0: True
[1. 1. 1. 1. 1.]


`fit_transform` is `fit` followed by `transform`. Every transformed column has a mean of 0 and a standard deviation of 1, which is what the scaler is for. `np.allclose` checks the means, since a mean that is zero up to rounding error can print as `-0.0`.

---

&#8592; **Back to:** [The Estimator API](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/01-the-estimator-api.ipynb)  &nbsp;&middot;&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
